In [ ]:
# Full Pipeline Runner (Databricks)
# One-click run: auto-generate seed CSVs -> run Bronze/Silver/Gold -> validation report.

import importlib.util
import subprocess
import sys
from pathlib import Path
from pyspark.sql import functions as F

dbutils.widgets.text(
    "repo_path",
    "/Workspace/Users/himanshu.kumar1@tothenew.com/databricks-medallion-pipeline",
    "Repo path",
)
dbutils.widgets.dropdown("regenerate_seed_data", "true", ["true", "false"], "Regenerate seed data")

repo_path = dbutils.widgets.get("repo_path").strip()
regenerate_seed_data = dbutils.widgets.get("regenerate_seed_data").strip().lower() == "true"

if not repo_path:
    raise ValueError("repo_path widget is required")

print(f"Using repo path: {repo_path}")

# -----------------------------------------------------------------------------
# Step 0: Generate seed CSVs directly in Databricks Volume (optional toggle)
# -----------------------------------------------------------------------------
volume_path = "/Volumes/workspace/default/medallion_data"
volume_path_uri = "dbfs:/Volumes/workspace/default/medallion_data"
local_seed_path = f"{repo_path}/data"
if regenerate_seed_data:
    if importlib.util.find_spec("faker") is None:
        print("Faker not found. Installing Faker==40.36.0 ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "Faker==40.36.0"])

    data_gen_path = f"{repo_path}/src/data_generation"
    if data_gen_path not in sys.path:
        sys.path.insert(0, data_gen_path)

    from generate_sample_data import main as generate_sample_data_main

    # Generate under /Workspace path, then copy to Volume via dbutils.fs.
    local_seed_dir = Path(local_seed_path)
    local_seed_dir.mkdir(parents=True, exist_ok=True)

    print(f"Generating CSV data into workspace path: {local_seed_path}")
    generate_exit_code = generate_sample_data_main(["--output-dir", local_seed_path])
    if int(generate_exit_code) != 0:
        raise RuntimeError(f"Data generation failed with exit code {generate_exit_code}")

    try:
        dbutils.fs.mkdirs(volume_path_uri)
    except Exception as exc:
        raise RuntimeError(
            "Unable to access target Volume path. Ensure volume exists at "
            "workspace.default.medallion_data before running full pipeline."
        ) from exc

    for filename in ["customers.csv", "products.csv", "orders.csv"]:
        src = f"file:{local_seed_path}/{filename}"
        dst = f"{volume_path_uri}/{filename}"
        dbutils.fs.cp(src, dst, True)

    print(f"Seed data generation completed and copied to: {volume_path}")
else:
    print("Skipping seed data generation (regenerate_seed_data=false).")

# -----------------------------------------------------------------------------
# Step 1: Run Bronze -> Silver -> Gold notebooks
# -----------------------------------------------------------------------------
notebooks = [
    f"{repo_path}/notebooks/bronze_runtime_validation",
    f"{repo_path}/notebooks/silver_runtime_validation",
    f"{repo_path}/notebooks/gold_runtime_validation",
]

stage_results = []
for nb in notebooks:
    print(f"Running: {nb}")
    try:
        result = dbutils.notebook.run(
            nb,
            timeout_seconds=3600,
            arguments={"repo_path": repo_path},
        )
        stage_results.append((nb.split("/")[-1], "PASS", str(result)))
    except Exception as exc:
        stage_results.append((nb.split("/")[-1], "FAIL", str(exc)))
        print("\n=== Stage Execution Summary ===")
        for stage_name, status, message in stage_results:
            print(f"{stage_name}: {status} | {message}")
        raise

print("\n=== Stage Execution Summary ===")
for stage_name, status, message in stage_results:
    print(f"{stage_name}: {status} | {message}")

# -----------------------------------------------------------------------------
# Step 2: Expected-vs-actual validation checks
# -----------------------------------------------------------------------------
checks = [
    ("bronze.customers row_count", 10000, spark.table("bronze.customers").count()),
    ("bronze.products row_count", 500, spark.table("bronze.products").count()),
    ("bronze.orders row_count", 100000, spark.table("bronze.orders").count()),
    ("silver.customers row_count", 10000, spark.table("silver.customers").count()),
    ("silver.products row_count", 500, spark.table("silver.products").count()),
    ("silver.orders row_count", 100000, spark.table("silver.orders").count()),
    ("gold.sales_by_product row_count", 500, spark.table("gold.sales_by_product").count()),
    ("gold.revenue_by_customer row_count", 9940, spark.table("gold.revenue_by_customer").count()),
    ("gold.customer_segmentation row_count", 4, spark.table("gold.customer_segmentation").count()),
    ("gold.daily_weekly_trends row_count", 2679, spark.table("gold.daily_weekly_trends").count()),
]

latest_metrics = spark.sql(
    """
    SELECT run_id, COUNT(*) AS metric_rows
    FROM silver.dq_metrics
    GROUP BY run_id
    ORDER BY MAX(run_timestamp) DESC
    LIMIT 1
    """
).collect()

if not latest_metrics:
    raise RuntimeError("No rows found in silver.dq_metrics. Silver validation report missing.")

checks.append(("silver.dq_metrics latest run rows", 10, int(latest_metrics[0]["metric_rows"])))

report_rows = []
for check_name, expected, actual in checks:
    status = "PASS" if expected == actual else "FAIL"
    report_rows.append((check_name, expected, actual, status))

report_df = spark.createDataFrame(
    report_rows,
    ["check_name", "expected", "actual", "status"],
)

print("\n=== Validation Report (Expected vs Actual) ===")
display(report_df.orderBy("status", "check_name"))

failed_count = report_df.filter(F.col("status") == "FAIL").count()
if failed_count > 0:
    raise RuntimeError(f"Validation failed: {failed_count} check(s) failed.")

print("\nAll validation checks passed.")
